# Food Price CPI — CFPR-Replica Backtesting Experiment

This notebook faithfully reproduces the **Canada's Food Price Report (CFPR)** evaluation
methodology and extends it with additional backtest years and probabilistic forecasts.

## Setup

The CFPR is typically published in November/December.  By that point, the July CPI
release is the most recent data in hand.  The report projects food prices **18 months
ahead** — to roughly January of the following year.  We replicate this by placing a
forecast origin at **July 1 of each year** and asking each predictor: *given all CPI
data available up to today, what will the index be in January 18 months from now?*

Running this for 16 consecutive years (July 2009 → July 2024) gives us a rich
retrospective picture: how well did each method work through a decade of low inflation,
a global pandemic, and the sharpest food price surge in forty years?

## How to use this notebook

**Change `CATEGORY_ID` in the configuration cell below** to run the same analysis for
any of the 9 food CPI categories.  Everything else adjusts automatically.

## Prerequisites

```bash
uv run python scripts/fetch_cpi.py     # populate StatCan CPI cache
uv run python scripts/fetch_fred.py    # requires FRED_API_KEY in .env
```

In [ ]:
from __future__ import annotations

import time
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from dotenv import load_dotenv

from aieng.forecasting.data import DataService, SeriesMetadata
from aieng.forecasting.data.adapters import FREDAdapter, StatCanAdapter
from aieng.forecasting.evaluation import (
    EvalTracker,
    MultiTargetBacktestSpec,
    MultiTargetEvalSpec,
    multi_backtest,
    multi_evaluate,
)
from methods.darts_arima import DartsAutoARIMAPredictor
from methods.darts_regression import (
    DartsLightGBMPredictor,
    DartsLinearRegressionPredictor,
)
from methods.naive import LastValuePredictor

# Notebook lives at implementations/experiments/food_price_forecasting/ —
# three levels below the repo root where data/ and reference_specs/ live.
ROOT = Path.cwd().resolve().parents[2]
load_dotenv(ROOT / ".env")

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)

## Configuration

Change `CATEGORY_ID` to any of the 9 series IDs to re-run the full analysis for that
food category.  The rest of the notebook reads from this single variable.

In [ ]:
# ── Change this to switch food CPI category ──────────────────────────────────
#
# Available series IDs (all from StatCan table 18-10-0004-11):
#   cpi_food_canada                      Overall Food (headline)
#   cpi_bakery_cereal_canada             Bakery and cereal products
#   cpi_dairy_eggs_canada                Dairy products and eggs
#   cpi_fish_seafood_canada              Fish, seafood and other marine products
#   cpi_restaurants_canada               Food purchased from restaurants
#   cpi_fruit_preparations_nuts_canada   Fruit, fruit preparations and nuts
#   cpi_meat_canada                      Meat
#   cpi_other_food_nonalcoholic_canada   Other food products and non-alcoholic beverages
#   cpi_vegetables_preparations_canada   Vegetables and vegetable preparations
#
CATEGORY_ID = "cpi_food_canada"   # <── change this line

# Human-readable labels and StatCan member filter strings.
_CATEGORY_META: dict[str, tuple[str, str]] = {
    "cpi_food_canada":                     ("Overall Food",                      "Food"),
    "cpi_bakery_cereal_canada":             ("Bakery and Cereal",                 "Bakery and cereal products (excluding baby food)"),
    "cpi_dairy_eggs_canada":                ("Dairy and Eggs",                    "Dairy products and eggs"),
    "cpi_fish_seafood_canada":              ("Fish and Seafood",                  "Fish, seafood and other marine products"),
    "cpi_restaurants_canada":              ("Food from Restaurants",              "Food purchased from restaurants"),
    "cpi_fruit_preparations_nuts_canada":  ("Fruit and Nuts",                    "Fruit, fruit preparations and nuts"),
    "cpi_meat_canada":                     ("Meat",                              "Meat"),
    "cpi_other_food_nonalcoholic_canada":  ("Other Food",                        "Other food products and non-alcoholic beverages"),
    "cpi_vegetables_preparations_canada":  ("Vegetables",                        "Vegetables and vegetable preparations"),
}

CATEGORY_PRETTY, STATCAN_LABEL = _CATEGORY_META[CATEGORY_ID]
print(f"Category: {CATEGORY_PRETTY}")
print(f"Series ID: {CATEGORY_ID}")
print(f"StatCan label: '{STATCAN_LABEL}'")

## 1. Data

We register the selected food CPI target and five FRED exogenous covariates with a
`DataService`.  Data is read from the local disk cache (populated by the fetch scripts)
— no network calls at notebook run time.

The `DataService` enforces a strict information cutoff: when the backtest harness calls
`svc.context(as_of=origin)`, only data with `timestamp <= origin` is visible to the
predictor.  This is the same discipline the CFPR team would have had — you can only
use data that existed on the day you made the forecast.

In [ ]:
FRED_COVARIATES: list[tuple[str, str, str, str]] = [
    ("fred_us_cpi_food_at_home",            "CPIFABSL",        "US CPI: Food at Home",              "Index 1982-84=100"),
    ("fred_us_cpi_meats_poultry_fish_eggs", "CUSR0000SAF112",  "US CPI: Meats, Poultry, Fish, Eggs", "Index 1982-84=100"),
    ("fred_us_cpi_fruits_vegetables",        "CUSR0000SAF113",  "US CPI: Fruits and Vegetables",      "Index 1982-84=100"),
    ("fred_canada_10yr_bond_yield",          "IRLTLT01CAM156N", "Canada 10-year bond yield",          "% per annum"),
    ("fred_canada_us_exchange_rate",         "EXCAUS",          "Canada / US exchange rate",          "CAD per USD"),
]
COVARIATE_IDS: list[str] = [c[0] for c in FRED_COVARIATES]

STATCAN_CACHE = ROOT / "data" / "statcan"
FRED_CACHE = ROOT / "data" / "fred"

svc = DataService()

svc.register(
    series_id=CATEGORY_ID,
    adapter=StatCanAdapter(
        table_id="18-10-0004-11",
        member_filter={"GEO": "Canada", "Products and product groups": STATCAN_LABEL},
        cache_dir=STATCAN_CACHE,
    ),
    metadata=SeriesMetadata(
        series_id=CATEGORY_ID,
        description=f"CPI {STATCAN_LABEL}, Canada (2002=100)",
        source="StatCan",
        units="Index 2002=100",
        frequency="MS",
        table_id="18-10-0004-11",
    ),
)

for series_id, fred_id, description, units in FRED_COVARIATES:
    svc.register(
        series_id=series_id,
        adapter=FREDAdapter(fred_id, cache_dir=FRED_CACHE),
        metadata=SeriesMetadata(
            series_id=series_id,
            description=description,
            source=f"FRED ({fred_id})",
            units=units,
            frequency="MS",
        ),
    )

summary = svc.summary().copy()
summary["start"] = summary["start"].dt.strftime("%Y-%m")
summary["end"] = summary["end"].dt.strftime("%Y-%m")
summary[["series_id", "description", "source", "frequency", "n_obs", "start", "end"]]

### Full history of the selected series

The index is reported as 2002 = 100.  Year-over-year percentage changes are derived
from the index at reporting time — the model always predicts the raw index level.

In [ ]:
today = datetime.now(tz=timezone.utc).replace(tzinfo=None)
target_df = svc.get_series(CATEGORY_ID, as_of=today)

print(f"{CATEGORY_ID}: {len(target_df):,} monthly observations")
print(f"  first: {target_df['timestamp'].iloc[0].date()}   value = {target_df['value'].iloc[0]:.2f}")
print(f"  last:  {target_df['timestamp'].iloc[-1].date()}  value = {target_df['value'].iloc[-1]:.2f}")
print()
print("Sample — first 3 rows:")
print(target_df.head(3).to_string(index=False))
print("...")
print("Sample — last 3 rows:")
print(target_df.tail(3).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(target_df["timestamp"], target_df["value"], lw=1.2, color="#1f4b99")
ax.set_title(f"Canada CPI — {CATEGORY_PRETTY} (2002=100)", fontsize=13)
ax.set_ylabel("Index value")
ax.grid(alpha=0.3)
ax.xaxis.set_major_locator(mdates.YearLocator(5))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
fig.tight_layout()
plt.show()

### Year-over-year change

The headline metric journalists and policymakers care about.  We model the index level,
then derive YoY change at the end — keeping the modelling problem simple and separable
from the reporting format.

In [ ]:
yoy = target_df.copy().set_index("timestamp")["value"]
yoy_pct = (yoy / yoy.shift(12) - 1.0) * 100.0

fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)
axes[0].plot(target_df["timestamp"], target_df["value"], lw=1.2, color="#1f4b99")
axes[0].set_ylabel("Index (2002=100)")
axes[0].set_title(f"Canada CPI — {CATEGORY_PRETTY}", fontsize=12)
axes[0].grid(alpha=0.3)

axes[1].bar(yoy_pct.index, yoy_pct.values, width=25, color=[
    "#d62728" if v > 0 else "#2ca02c" for v in yoy_pct.values
], alpha=0.7)
axes[1].axhline(0, color="black", lw=0.8)
axes[1].set_ylabel("YoY change (%)")
axes[1].set_title("Year-over-year % change (index level – 12 months prior)")
axes[1].grid(alpha=0.3, axis="y")
axes[1].xaxis.set_major_locator(mdates.YearLocator(5))
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
fig.tight_layout()
plt.show()

## 2. Experiment design

### Replicating the CFPR methodology

Each July we pretend we have just received the freshest available CPI data.  We fit
each model on that data, project 18 months ahead, then check the actual outcome once
the resolution date arrives.  Running this for 16 consecutive Julys (2009–2024) gives
a meaningful sample of retrospective forecast errors across very different macro
environments.

The spec is loaded from `reference_specs/food_cpi/food_cpi_18m_backtest.yaml`.
To run the analysis on a single target category, we filter the spec to the task whose
`target_series_id` matches `CATEGORY_ID`.

In [ ]:
with open(ROOT / "reference_specs" / "food_cpi" / "food_cpi_18m_backtest.yaml") as f:
    full_backtest_spec = MultiTargetBacktestSpec.model_validate(yaml.safe_load(f))

with open(ROOT / "reference_specs" / "food_cpi" / "food_cpi_18m_eval.yaml") as f:
    full_eval_spec = MultiTargetEvalSpec.model_validate(yaml.safe_load(f))

# Filter multi-target specs to the single selected category.
backtest_spec = full_backtest_spec.model_copy(
    update={"tasks": [t for t in full_backtest_spec.tasks if t.target_series_id == CATEGORY_ID]}
)
eval_spec = full_eval_spec.model_copy(
    update={"tasks": [t for t in full_eval_spec.tasks if t.target_series_id == CATEGORY_ID]}
)

origins = backtest_spec.specs()[0].origins()
eval_origins = eval_spec.specs()[0].origins()

print(f"Backtest: {len(origins)} annual July origins")
for o in origins:
    fd = (pd.Timestamp(o) + pd.DateOffset(months=18)).date()
    tag = " ← protected eval" if o in eval_origins else ""
    print(f"  {o.date()}  →  forecast date {fd}{tag}")

### Timeline visualisation

The grey curve is the full observed series.  Each **green tick** at the bottom marks
a backtest origin (July of that year); the **orange tick** marks the four origins
reserved for the protected eval window.  The annotated arrow shows one example of what
a predictor is asked to do at origin Jul 2020: given everything up to that date,
predict the index at Jan 2022 — 18 months later.

In [ ]:
def plot_timeline() -> None:
    series = svc.get_series(CATEGORY_ID, as_of=today)
    fig, ax = plt.subplots(figsize=(13, 4.5))
    ax.plot(series["timestamp"], series["value"], color="#666", lw=1.0)

    ax.axvspan(backtest_spec.start, backtest_spec.end,
               color="#2ca02c", alpha=0.07, label="backtest window (2009–2024)")
    ax.axvspan(eval_spec.start, eval_spec.end,
               color="#ff7f0e", alpha=0.18, label="protected eval window (2021–2024)")

    y_lo = series["value"].min() * 0.97
    for o in origins:
        color = "#ff7f0e" if o in eval_origins else "#2ca02c"
        ax.plot([o, o], [y_lo, y_lo * 1.015], color=color, lw=1.4)

    # Annotate a sample origin → forecast_date arrow.
    sample_origin = pd.Timestamp("2020-07-01")
    sample_fd = sample_origin + pd.DateOffset(months=18)
    v_origin = float(series.loc[series["timestamp"] == sample_origin, "value"].iloc[0])
    v_fd = float(series.loc[series["timestamp"] == sample_fd, "value"].iloc[0])
    ax.annotate("", xy=(sample_fd, v_fd), xytext=(sample_origin, v_origin),
                arrowprops={"arrowstyle": "->", "color": "#d62728", "lw": 1.8})
    ax.scatter([sample_origin, sample_fd], [v_origin, v_fd],
               color="#d62728", zorder=5, s=28)
    ax.annotate("as_of = Jul 2020", xy=(sample_origin, v_origin),
                xytext=(-6, -22), textcoords="offset points", fontsize=9, color="#d62728")
    ax.annotate("forecast_date = Jan 2022", xy=(sample_fd, v_fd),
                xytext=(6, 8), textcoords="offset points", fontsize=9, color="#d62728")

    ax.set_title(f"{CATEGORY_PRETTY} — experiment timeline: 16 July origins, 18-month horizon")
    ax.set_ylabel("Index (2002=100)")
    ax.grid(alpha=0.3)
    ax.legend(loc="upper left")
    ax.xaxis.set_major_locator(mdates.YearLocator(5))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    fig.tight_layout()
    plt.show()


plot_timeline()

## 3. Predictors

Four methods are compared.  All produce probabilistic forecasts (a distribution,
not just a point), so CRPS is a fair common metric.

| # | `predictor_id` | Model family | Covariates |
|---|----------------|-------------|------------|
| 1 | `last_value` | Naïve — last observed value carried forward | — |
| 2 | `darts_autoarima` | Auto-selected ARIMA (order search via AIC) | — |
| 3 | `darts_linreg_cov` | Quantile linear regression on lagged features | 5 FRED series |
| 4 | `darts_lightgbm_cov` | Gradient-boosted quantile regression | 5 FRED series |

Predictors 3 and 4 see the same FRED covariates (US food CPI sub-indices, Canadian
bond yield, and CAD/USD exchange rate) lagged by up to 12 months.  The covariate
history is also cut off at `as_of`, matching the information discipline of the target.

In [ ]:
predictors = [
    LastValuePredictor(),
    DartsAutoARIMAPredictor(),
    DartsLinearRegressionPredictor(
        lags=12,
        lags_past_covariates=12,
        covariate_series_ids=COVARIATE_IDS,
    ),
    DartsLightGBMPredictor(
        lags=12,
        lags_past_covariates=12,
        covariate_series_ids=COVARIATE_IDS,
    ),
]

PREDICTOR_COLORS: dict[str, str] = {
    "last_value":           "#7f7f7f",
    "darts_autoarima":      "#1f77b4",
    "darts_linreg_cov":     "#2ca02c",
    "darts_lightgbm_cov":   "#d62728",
}

for p in predictors:
    print(f"  {p.predictor_id:<22}  {type(p).__name__}")

## 4. Backtest

Each predictor is fit and evaluated at every one of the 16 July origins.  This means
**16 full model refits per predictor** — one per year, each time consuming all available
history up to that July.  The result is 16 probabilistic forecasts per predictor, each
scored against the actual CPI value 18 months later.

In [ ]:
backtest_results: dict[str, object] = {}
task_id = backtest_spec.tasks[0].task_id

for predictor in predictors:
    t0 = time.perf_counter()
    results = multi_backtest(predictor=predictor, spec=backtest_spec, data_service=svc)
    elapsed = time.perf_counter() - t0
    result = results[task_id]
    backtest_results[predictor.predictor_id] = result
    print(f"  {predictor.predictor_id:<22}  origins={len(result.predictions)}  "
          f"mean CRPS={result.mean_crps:6.3f}  ({elapsed:.1f}s)")

## 5. Disaggregated errors — year by year

The most important diagnostic.  For each July origin we show the **actual** index value
at the 18-month resolution date alongside the **median forecast** and **80% prediction
interval** from every predictor.  Errors that stick out immediately — e.g. 2020 → 2022
(COVID) — tell us a lot about where each method failed and why.

In [ ]:
def resolve_actual(series_id: str, when: datetime) -> float | None:
    df = svc.get_series(series_id, as_of=today)
    match = df[df["timestamp"] == pd.Timestamp(when)]
    return float(match["value"].iloc[0]) if not match.empty else None


# Collect actuals and forecasts per origin.
origin_years: list[int] = []
actuals: list[float] = []
forecasts: dict[str, list[dict]] = {p.predictor_id: [] for p in predictors}

for pred in backtest_results[predictors[0].predictor_id].predictions:
    actual = resolve_actual(CATEGORY_ID, pred.forecast_date)
    if actual is not None:
        origin_years.append(pred.as_of.year)
        actuals.append(actual)

for predictor_id, result in backtest_results.items():
    for pred in result.predictions:
        actual = resolve_actual(CATEGORY_ID, pred.forecast_date)
        if actual is None:
            continue
        forecasts[predictor_id].append({
            "origin_year": pred.as_of.year,
            "forecast_date": pred.forecast_date,
            "median": pred.payload.point_forecast,
            "q10": pred.payload.quantiles[0.10],
            "q20": pred.payload.quantiles[0.20],
            "q80": pred.payload.quantiles[0.80],
            "q90": pred.payload.quantiles[0.90],
            "actual": actual,
        })

print(f"Origins with resolved actuals: {len(origin_years)}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

x = np.arange(len(origin_years))
x_labels = [str(y) for y in origin_years]

# ── Panel 1: index level — actuals vs. median forecasts ───────────────────────
ax = axes[0]
ax.scatter(x, actuals, color="black", zorder=6, s=40, label="actual", marker="D")

n_preds = len(predictors)
offsets = np.linspace(-0.25, 0.25, n_preds)
for i, predictor in enumerate(predictors):
    pid = predictor.predictor_id
    color = PREDICTOR_COLORS[pid]
    rows = forecasts[pid]
    xi = x[:len(rows)] + offsets[i]
    medians = [r["median"] for r in rows]
    q10 = [r["q10"] for r in rows]
    q90 = [r["q90"] for r in rows]
    q20 = [r["q20"] for r in rows]
    q80 = [r["q80"] for r in rows]
    ax.vlines(xi, q10, q90, color=color, lw=5, alpha=0.25)
    ax.vlines(xi, q20, q80, color=color, lw=5, alpha=0.50)
    ax.scatter(xi, medians, color=color, s=20, zorder=5)

ax.set_ylabel("CPI index (2002=100)")
ax.set_title(f"{CATEGORY_PRETTY} — actual vs. 18-month-ahead forecasts by July origin year")
ax.grid(alpha=0.3, axis="y")
legend_handles = [mpatches.Patch(color=PREDICTOR_COLORS[p.predictor_id], label=p.predictor_id)
                  for p in predictors]
legend_handles.insert(0, plt.scatter([], [], color="black", marker="D", label="actual"))
ax.legend(handles=legend_handles, fontsize=9, ncol=5)

# ── Panel 2: median forecast error (forecast – actual) ────────────────────────
ax2 = axes[1]
ax2.axhline(0, color="black", lw=0.8, ls="--")
for i, predictor in enumerate(predictors):
    pid = predictor.predictor_id
    color = PREDICTOR_COLORS[pid]
    rows = forecasts[pid]
    xi = x[:len(rows)] + offsets[i]
    errors = [r["median"] - r["actual"] for r in rows]
    ax2.bar(xi, errors, width=0.18, color=color, alpha=0.8, label=pid)

ax2.set_ylabel("Median error (forecast − actual)")
ax2.set_title("Median forecast error by year — positive = over-prediction, negative = under-prediction")
ax2.set_xticks(x)
ax2.set_xticklabels(x_labels, rotation=45, ha="right")
ax2.set_xlabel("Forecast origin year (July)")
ax2.grid(alpha=0.3, axis="y")

fig.tight_layout()
plt.show()

## 6. Error statistics summary

### CRPS per year and overall

CRPS scores the full predictive distribution against the observed outcome.  Lower is
better.  A narrow, well-centred distribution earns a low score; a wide or mis-centred
one earns a high score.

In [ ]:
crps_rows: dict[str, dict] = {}
for predictor in predictors:
    pid = predictor.predictor_id
    result = backtest_results[pid]
    row = {}
    for pred, score in zip(result.predictions, result.scores):
        row[str(pred.as_of.year)] = score
    row["mean CRPS"] = result.mean_crps
    row["std"] = float(np.std(result.scores))
    row["best yr"] = min(row, key=lambda k: row[k] if k not in ("mean CRPS", "std", "best yr", "worst yr") else float("inf"))
    row["worst yr"] = max(row, key=lambda k: row[k] if k not in ("mean CRPS", "std", "best yr", "worst yr") else float("-inf"))
    crps_rows[pid] = row

crps_df = pd.DataFrame(crps_rows).T.sort_values("mean CRPS")
crps_df.style.format(
    {c: "{:.3f}" for c in crps_df.columns if c not in ("best yr", "worst yr")}
).background_gradient(cmap="RdYlGn_r", axis=None,
    subset=[c for c in crps_df.columns if c not in ("std", "mean CRPS", "best yr", "worst yr")])

### MAPE on the median forecast

Mean Absolute Percentage Error computed on the median (point) forecast.  Familiar from
the original CFPR exercise.  Reported here as a secondary diagnostic, not the ranking
criterion.

In [ ]:
mape_rows: dict[str, dict] = {}
for predictor in predictors:
    pid = predictor.predictor_id
    row: dict[str, float | str] = {}
    apes: list[float] = []
    for pred_rec in backtest_results[pid].predictions:
        actual = resolve_actual(CATEGORY_ID, pred_rec.forecast_date)
        if actual is None or actual == 0:
            continue
        ape = abs(pred_rec.payload.point_forecast - actual) / abs(actual) * 100.0
        row[str(pred_rec.as_of.year)] = ape
        apes.append(ape)
    row["mean MAPE"] = float(np.mean(apes))
    row["std"] = float(np.std(apes))
    row["best yr"] = min((k for k in row if k not in ("mean MAPE", "std", "best yr", "worst yr")),
                         key=lambda k: row[k])  # type: ignore[arg-type]
    row["worst yr"] = max((k for k in row if k not in ("mean MAPE", "std", "best yr", "worst yr")),
                          key=lambda k: row[k])  # type: ignore[arg-type]
    mape_rows[pid] = row

mape_df = pd.DataFrame(mape_rows).T.sort_values("mean MAPE")
mape_df.style.format(
    {c: "{:.2f}%" for c in mape_df.columns if c not in ("best yr", "worst yr")}
).background_gradient(cmap="RdYlGn_r", axis=None,
    subset=[c for c in mape_df.columns if c not in ("std", "mean MAPE", "best yr", "worst yr")])

### MAPE distribution — visualised

Box plots and swarm-style jitter show the full distribution of annual MAPE values,
not just the mean.  Outlier years are immediately visible.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

mape_data = {}
for predictor in predictors:
    pid = predictor.predictor_id
    vals = []
    for pred_rec in backtest_results[pid].predictions:
        actual = resolve_actual(CATEGORY_ID, pred_rec.forecast_date)
        if actual and actual != 0:
            vals.append(abs(pred_rec.payload.point_forecast - actual) / abs(actual) * 100.0)
    mape_data[pid] = vals

positions = list(range(len(predictors)))
bp = ax.boxplot(
    [mape_data[p.predictor_id] for p in predictors],
    positions=positions,
    widths=0.5,
    patch_artist=True,
    medianprops={"color": "black", "lw": 2},
)
for patch, predictor in zip(bp["boxes"], predictors):
    patch.set_facecolor(PREDICTOR_COLORS[predictor.predictor_id])
    patch.set_alpha(0.6)

# Overlay individual year dots with jitter.
rng = np.random.default_rng(42)
for i, predictor in enumerate(predictors):
    vals = mape_data[predictor.predictor_id]
    jitter = rng.uniform(-0.12, 0.12, size=len(vals))
    ax.scatter(i + jitter, vals, color=PREDICTOR_COLORS[predictor.predictor_id],
               s=18, zorder=5, alpha=0.9, edgecolors="white", linewidths=0.4)

ax.set_xticks(positions)
ax.set_xticklabels([p.predictor_id for p in predictors], rotation=15, ha="right")
ax.set_ylabel("MAPE on median forecast (%)")
ax.set_title(f"{CATEGORY_PRETTY} — distribution of annual MAPE values (each dot = one July origin)")
ax.grid(alpha=0.3, axis="y")
fig.tight_layout()
plt.show()

## 7. Year-over-year confidence ranges

The CFPR delivers forecasts as **year-over-year percentage change ranges**.  We derive
these from the most recent backtest origin (July 2024 → Jan 2026) using the best
predictor by CRPS:

- The **point estimate** is the median index forecast converted to YoY %.
- The **80% interval** uses the 10th and 90th quantile index forecasts converted
  to YoY % against the index value from January 2025 (12 months before the
  forecast date).

In [ ]:
best_pid = sorted(backtest_results, key=lambda pid: backtest_results[pid].mean_crps)[0]
best_result = backtest_results[best_pid]
latest_pred = best_result.predictions[-1]

forecast_date = pd.Timestamp(latest_pred.forecast_date)
baseline_date = forecast_date - pd.DateOffset(months=12)
baseline_val = resolve_actual(CATEGORY_ID, baseline_date.to_pydatetime())

q = latest_pred.payload.quantiles
median_idx = latest_pred.payload.point_forecast

def idx_to_yoy(idx_val: float) -> float:
    assert baseline_val is not None
    return (idx_val - baseline_val) / baseline_val * 100.0

print(f"Best predictor: {best_pid}")
print(f"Origin:         {latest_pred.as_of.date()} (July 2024)")
print(f"Forecast date:  {forecast_date.date()} (January 2026)")
print(f"Baseline date:  {baseline_date.date()} (January 2025, for YoY)")
print()
print(f"CPI index forecast:")
print(f"  Median:    {median_idx:.2f}   (baseline: {baseline_val:.2f})")
print(f"  Q10–Q90:   [{q[0.10]:.2f}, {q[0.90]:.2f}]")
print()
print(f"YoY % change forecast (Jan 2026 vs Jan 2025):")
print(f"  Point estimate:   {idx_to_yoy(median_idx):+.1f}%")
print(f"  80% interval:     {idx_to_yoy(q[0.10]):+.1f}% to {idx_to_yoy(q[0.90]):+.1f}%")

## 8. Protected eval

Finally, we run the best predictor on the protected eval window (four July origins:
2021–2024) with a budget tracker.  One call to `multi_evaluate()` consumes one budget
slot against `max_runs=5`.  Comparing eval CRPS to backtest CRPS shows whether
performance on the open window generalised.

In [ ]:
tracker_path = Path.cwd() / "eval_runs.yaml"
tracker = EvalTracker(tracker_path)
winning_predictor = next(p for p in predictors if p.predictor_id == best_pid)

runs_used = tracker.runs_for(eval_spec.spec_id)
print(f"Tracker file: {tracker_path.name}")
print(f"Spec:         {eval_spec.spec_id}")
print(f"Runs used:    {runs_used} / {eval_spec.max_runs}")
print(f"Predictor:    {winning_predictor.predictor_id}")

In [ ]:
eval_results = multi_evaluate(
    predictor=winning_predictor,
    spec=eval_spec,
    data_service=svc,
    tracker=tracker,
)

eval_task_id = eval_spec.tasks[0].task_id
eval_result = eval_results[eval_task_id]

print(f"Eval origins scored: {len(eval_result.predictions)}")
print(f"Eval mean CRPS:      {eval_result.mean_crps:.3f}")
print(f"Backtest mean CRPS:  {best_result.mean_crps:.3f}")
print(f"Run number:          {eval_result.run_number} / {eval_spec.max_runs}")
print()
print("Per-origin eval breakdown:")
for pred, score in zip(eval_result.predictions, eval_result.scores):
    actual = resolve_actual(CATEGORY_ID, pred.forecast_date)
    err = f"  error = {pred.payload.point_forecast - actual:+.2f}" if actual else ""
    print(f"  {pred.as_of.date()} → {pred.forecast_date.date()}   CRPS={score:.3f}{err}")